# 🍷 Drinks Quality Prediction System
### End-to-End ML Project — Google Colab Notebook

**Pipeline Stages:**
1. Environment Setup
2. Project Structure Creation
3. Data Ingestion
4. Data Validation
5. Data Transformation
6. Model Training
7. Model Evaluation
8. Prediction
9. Flask App (Optional)

> ⚠️ **Note:** Run cells in order. All stages are sequential and depend on prior stages.

---
## 🔧 Section 1 — Install Dependencies

In [1]:
!pip install -q scikit-learn pandas numpy joblib flask mlflow dagshub pyyaml ensure python-box

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 121.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

---
## 🔐 Section 2 — Environment Variables (Secrets)

In [2]:
import os
from google.colab import userdata

# ── MongoDB (from Colab Secrets) ──────────────────────────────────────────────
os.environ['MONGO_DB_URL'] = userdata.get('MONGO_DB_URL')

# ── MLflow / DagsHub ──────────────────────────────────────────────────────────
USE_DAGSHUB = True  # ✅ Set to False to use local MLflow

if USE_DAGSHUB:
    os.environ['MLFLOW_TRACKING_URI']      = userdata.get('MLFLOW_TRACKING_URI')
    os.environ['MLFLOW_TRACKING_USERNAME'] = userdata.get('MLFLOW_TRACKING_USERNAME')
    os.environ['MLFLOW_TRACKING_PASSWORD'] = userdata.get('MLFLOW_TRACKING_PASSWORD')
else:
    # Local MLflow — logs saved inside Colab
    os.environ['MLFLOW_TRACKING_URI'] = f"file://{os.getcwd()}/mlruns"

print('✅ Env vars set.')
print(f"   MLFLOW_TRACKING_URI = {os.environ['MLFLOW_TRACKING_URI']}")

✅ Env vars set.
   MLFLOW_TRACKING_URI = https://dagshub.com/prithusarkar90/networksecurity.mlflow


---
## 📁 Section 3 — Project Structure Creation

Creates the full `mlProject` package folder structure (equivalent to running `template.py`).

In [3]:
import os
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO, format='[%(asctime)s]: %(message)s:')

project_name = "mlProject"

list_of_files = [
    f"src/{project_name}/__init__.py",
    f"src/{project_name}/components/__init__.py",
    f"src/{project_name}/utils/__init__.py",
    f"src/{project_name}/utils/common.py",
    f"src/{project_name}/config/__init__.py",
    f"src/{project_name}/config/configuration.py",
    f"src/{project_name}/pipeline/__init__.py",
    f"src/{project_name}/entity/__init__.py",
    f"src/{project_name}/entity/config_entity.py",
    f"src/{project_name}/constants/__init__.py",
    f"src/{project_name}/components/data_ingestion.py",
    f"src/{project_name}/components/data_validation.py",
    f"src/{project_name}/components/data_transformation.py",
    f"src/{project_name}/components/model_trainer.py",
    f"src/{project_name}/components/model_evaluation.py",
    f"src/{project_name}/pipeline/stage_01_data_ingestion.py",
    f"src/{project_name}/pipeline/stage_02_data_validation.py",
    f"src/{project_name}/pipeline/stage_03_data_transformation.py",
    f"src/{project_name}/pipeline/stage_04_model_trainer.py",
    f"src/{project_name}/pipeline/stage_05_model_evaluation.py",
    f"src/{project_name}/pipeline/prediction.py",
    "config/config.yaml",
    "params.yaml",
    "schema.yaml",
    "main.py",
    "app.py",
    "setup.py",
    "templates/index.html",
    "templates/results.html",
]

for filepath in list_of_files:
    filepath = Path(filepath)
    filedir, filename = os.path.split(filepath)
    if filedir != "":
        os.makedirs(filedir, exist_ok=True)
    if (not os.path.exists(filepath)) or (os.path.getsize(filepath) == 0):
        with open(filepath, "w") as f:
            pass
        logging.info(f"Creating empty file: {filepath}")
    else:
        logging.info(f"{filename} already exists")

print("✅ Project structure created.")

✅ Project structure created.


---
## ✍️ Section 4 — Write All Source Files

Writes every module of the `mlProject` package directly into the created structure.

### 4.1 — `setup.py`

In [4]:
setup_py = '''
import setuptools

with open("README.md", "r", encoding="utf-8") as f:
    long_description = f.read()

__version__ = "0.0.0"

REPO_NAME = "End-to-end-ML-Project"
AUTHOR_USER_NAME = "entbappy"
SRC_REPO = "mlProject"
AUTHOR_EMAIL = "entbappy73@gmail.com"

setuptools.setup(
    name=SRC_REPO,
    version=__version__,
    author=AUTHOR_USER_NAME,
    author_email=AUTHOR_EMAIL,
    description="A small python package for ml app",
    long_description=long_description,
    long_description_content="text/markdown",
    url=f"https://github.com/{AUTHOR_USER_NAME}/{REPO_NAME}",
    project_urls={
        "Bug Tracker": f"https://github.com/{AUTHOR_USER_NAME}/{REPO_NAME}/issues",
    },
    package_dir={"": "src"},
    packages=setuptools.find_packages(where="src")
)
'''

# Create a minimal README so setup.py doesn't fail
with open("README.md", "w") as f:
    f.write("# Drinks Quality Prediction\n")

with open("setup.py", "w") as f:
    f.write(setup_py.strip())

print("✅ setup.py written.")

✅ setup.py written.


### 4.2 — `src/mlProject/__init__.py` (Logger)

In [5]:
init_py = '''
import os
import sys
import logging

logging_str = "[%(asctime)s: %(levelname)s: %(module)s: %(message)s]"
log_dir = "logs"
log_filepath = os.path.join(log_dir, "running_logs.log")
os.makedirs(log_dir, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format=logging_str,
    handlers=[
        logging.FileHandler(log_filepath),
        logging.StreamHandler(sys.stdout)
    ]
)

logger = logging.getLogger("mlProjectLogger")
'''

with open("src/mlProject/__init__.py", "w") as f:
    f.write(init_py.strip())

print("✅ __init__.py (logger) written.")

✅ __init__.py (logger) written.


### 4.3 — `src/mlProject/constants/__init__.py`

In [6]:
constants_py = '''
from pathlib import Path

CONFIG_FILE_PATH = Path("config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")
SCHEMA_FILE_PATH = Path("schema.yaml")
'''

with open("src/mlProject/constants/__init__.py", "w") as f:
    f.write(constants_py.strip())

print("✅ constants/__init__.py written.")

✅ constants/__init__.py written.


### 4.4 — `src/mlProject/utils/common.py`

In [7]:
common_py = '''
import os
import yaml
import json
import joblib
from ensure import ensure_annotations
from box import ConfigBox
from pathlib import Path
from typing import Any
from mlProject import logger


@ensure_annotations
def read_yaml(path_to_yaml: Path) -> ConfigBox:
    """Reads a yaml file and returns a ConfigBox."""
    try:
        with open(path_to_yaml) as yaml_file:
            content = yaml.safe_load(yaml_file)
            logger.info(f"yaml file: {path_to_yaml} loaded successfully")
            return ConfigBox(content)
    except Exception as e:
        raise e


@ensure_annotations
def create_directories(path_to_directories: list, verbose=True):
    """Creates list of directories."""
    for path in path_to_directories:
        os.makedirs(path, exist_ok=True)
        if verbose:
            logger.info(f"Created directory at: {path}")


@ensure_annotations
def save_json(path: Path, data: dict):
    """Saves dict as JSON."""
    with open(path, "w") as f:
        json.dump(data, f, indent=4)
    logger.info(f"json file saved at: {path}")


@ensure_annotations
def load_json(path: Path) -> ConfigBox:
    """Loads JSON file and returns a ConfigBox."""
    with open(path) as f:
        content = json.load(f)
    logger.info(f"json file loaded successfully from: {path}")
    return ConfigBox(content)


@ensure_annotations
def save_bin(data: Any, path: Path):
    """Saves data as binary using joblib."""
    joblib.dump(value=data, filename=path)
    logger.info(f"Binary file saved at: {path}")


@ensure_annotations
def load_bin(path: Path) -> Any:
    """Loads binary data."""
    data = joblib.load(path)
    logger.info(f"Binary file loaded from: {path}")
    return data


@ensure_annotations
def get_size(path: Path) -> str:
    """Gets size of file in KB."""
    size_in_kb = round(os.path.getsize(path) / 1024)
    return f"~ {size_in_kb} KB"
'''

with open("src/mlProject/utils/common.py", "w") as f:
    f.write(common_py.strip())

print("✅ utils/common.py written.")

✅ utils/common.py written.


### 4.5 — `src/mlProject/entity/config_entity.py`

In [8]:
config_entity_py = '''
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path


@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    unzip_data_dir: Path
    all_schema: dict


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
'''

with open("src/mlProject/entity/config_entity.py", "w") as f:
    f.write(config_entity_py.strip())

print("✅ entity/config_entity.py written.")

✅ entity/config_entity.py written.


### 4.6 — `config/config.yaml`, `params.yaml`, `schema.yaml`

In [9]:
config_yaml = """artifacts_root: artifacts

data_ingestion:
  root_dir: artifacts/data_ingestion
  source_URL: https://github.com/entbappy/Branching-tutorial/raw/refs/heads/master/Drinks-data.zip
  local_data_file: artifacts/data_ingestion/data.zip
  unzip_dir: artifacts/data_ingestion

data_validation:
  root_dir: artifacts/data_validation
  unzip_data_dir: artifacts/data_ingestion/Drinks-data.csv
  STATUS_FILE: artifacts/data_validation/status.txt

data_transformation:
  root_dir: artifacts/data_transformation
  data_path: artifacts/data_ingestion/Drinks-data.csv

model_trainer:
  root_dir: artifacts/model_trainer
  train_data_path: artifacts/data_transformation/train.csv
  test_data_path: artifacts/data_transformation/test.csv
  model_name: model.joblib

model_evaluation:
  root_dir: artifacts/model_evaluation
  test_data_path: artifacts/data_transformation/test.csv
  model_path: artifacts/model_trainer/model.joblib
  metric_file_name: artifacts/model_evaluation/metrics.json
"""

params_yaml = """ElasticNet:
  alpha: 0.2
  l1_ratio: 0.1
"""

schema_yaml = """COLUMNS:
  fixed acidity: float64
  volatile acidity: float64
  citric acid: float64
  residual sugar: float64
  chlorides: float64
  free sulfur dioxide: float64
  total sulfur dioxide: float64
  density: float64
  pH: float64
  sulphates: float64
  alcohol: float64
  quality: int64

TARGET_COLUMN:
  name: quality
"""

with open("config/config.yaml", "w") as f:
    f.write(config_yaml)

with open("params.yaml", "w") as f:
    f.write(params_yaml)

with open("schema.yaml", "w") as f:
    f.write(schema_yaml)

print("✅ config.yaml, params.yaml, schema.yaml written.")

✅ config.yaml, params.yaml, schema.yaml written.


### 4.7 — `src/mlProject/config/configuration.py`

In [10]:
configuration_py = '''
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories
from mlProject.entity.config_entity import (
    DataIngestionConfig,
    DataValidationConfig,
    DataTransformationConfig,
    ModelTrainerConfig,
    ModelEvaluationConfig
)


class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        return DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS
        create_directories([config.root_dir])
        return DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            unzip_data_dir=config.unzip_data_dir,
            all_schema=schema,
        )

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir])
        return DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
        )

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN
        create_directories([config.root_dir])
        return ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            model_name=config.model_name,
            alpha=params.alpha,
            l1_ratio=params.l1_ratio,
            target_column=schema.name
        )

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN
        create_directories([config.root_dir])
        return ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metric_file_name=config.metric_file_name,
            target_column=schema.name
        )
'''

with open("src/mlProject/config/configuration.py", "w") as f:
    f.write(configuration_py.strip())

print("✅ config/configuration.py written.")

✅ config/configuration.py written.


### 4.8 — Components

In [12]:
from pathlib import Path

src = """import os
import urllib.request as request
import zipfile
from mlProject import logger
from mlProject.utils.common import get_size
from pathlib import Path
from mlProject.entity.config_entity import DataIngestionConfig

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file)
            logger.info(f"{filename} downloaded!")
        else:
            logger.info(f"File exists: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        os.makedirs(self.config.unzip_dir, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as z:
            z.extractall(self.config.unzip_dir)
"""
Path("src/mlProject/components/data_ingestion.py").write_text(src)
print("✅ components/data_ingestion.py written.")

✅ components/data_ingestion.py written.


In [14]:
from pathlib import Path

src = """import os
from mlProject import logger
import pandas as pd
from mlProject.entity.config_entity import DataValidationConfig

class DataValiadtion:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self) -> bool:
        try:
            data = pd.read_csv(self.config.unzip_data_dir)
            all_cols = list(data.columns)
            all_schema = self.config.all_schema.keys()
            validation_status = None
            for col in all_cols:
                if col not in all_schema:
                    validation_status = False
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f'Validation status: {validation_status}')
                else:
                    validation_status = True
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f'Validation status: {validation_status}')
            return validation_status
        except Exception as e:
            raise e
"""
Path("src/mlProject/components/data_validation.py").write_text(src)
print("✅ components/data_validation.py written.")

✅ components/data_validation.py written.


In [16]:
from pathlib import Path

src = """import os
from mlProject import logger
from sklearn.model_selection import train_test_split
import pandas as pd
from mlProject.entity.config_entity import DataTransformationConfig

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def train_test_spliting(self):
        data = pd.read_csv(self.config.data_path)
        train, test = train_test_split(data)
        train.to_csv(os.path.join(self.config.root_dir, 'train.csv'), index=False)
        test.to_csv(os.path.join(self.config.root_dir, 'test.csv'), index=False)
        logger.info('Split data into training and test sets')
        logger.info(train.shape)
        logger.info(test.shape)
        print(train.shape)
        print(test.shape)
"""
Path("src/mlProject/components/data_transformation.py").write_text(src)
print("✅ components/data_transformation.py written.")

✅ components/data_transformation.py written.


In [18]:
from pathlib import Path

src = """import pandas as pd
import os
from mlProject import logger
from sklearn.linear_model import ElasticNet
import joblib
from mlProject.entity.config_entity import ModelTrainerConfig

class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data  = pd.read_csv(self.config.test_data_path)
        train_x = train_data.drop([self.config.target_column], axis=1)
        test_x  = test_data.drop([self.config.target_column], axis=1)
        train_y = train_data[[self.config.target_column]]
        test_y  = test_data[[self.config.target_column]]
        lr = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio, random_state=42)
        lr.fit(train_x, train_y)
        joblib.dump(lr, os.path.join(self.config.root_dir, self.config.model_name))
"""
Path("src/mlProject/components/model_trainer.py").write_text(src)
print("✅ components/model_trainer.py written.")

✅ components/model_trainer.py written.


In [33]:
from pathlib import Path

src = """import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import joblib
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from pathlib import Path
from mlProject.entity.config_entity import ModelEvaluationConfig
from mlProject.utils.common import save_json
from mlProject import logger

class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae  = mean_absolute_error(actual, pred)
        r2   = r2_score(actual, pred)
        return rmse, mae, r2

    def log_into_mlflow(self): # Added 'self' here
        test_data = pd.read_csv(self.config.test_data_path)
        model     = joblib.load(self.config.model_path)
        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]
        mlflow.set_tracking_uri(os.environ.get('MLFLOW_TRACKING_URI', 'mlruns'))
        with mlflow.start_run():
            predicted = model.predict(test_x)
            (rmse, mae, r2) = self.eval_metrics(test_y, predicted)
            save_json(path=Path(self.config.metric_file_name), data={'rmse': rmse, 'mae': mae, 'r2': r2})
            mlflow.log_params(self.config.all_params)
            mlflow.log_metric('rmse', rmse)
            mlflow.log_metric('r2', r2)
            mlflow.log_metric('mae', mae)
            sig = infer_signature(test_x, model.predict(test_x))
            mlflow.sklearn.log_model(sk_model=model, artifact_path='model',
                signature=sig, registered_model_name='ElasticnetModel')
            logger.info(f'RMSE: {rmse}  MAE: {mae}  R2: {r2}')

    def save_results(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model     = joblib.load(self.config.model_path)
        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]
        predicted = model.predict(test_x)
        (rmse, mae, r2) = self.eval_metrics(test_y, predicted)
        save_json(path=Path(self.config.metric_file_name), data={'rmse': rmse, 'mae': mae, 'r2': r2})
"""
Path("src/mlProject/components/model_evaluation.py").write_text(src)
print("✅ components/model_evaluation.py written.")

✅ components/model_evaluation.py written.


### 4.9 — Pipeline Stages

In [22]:
from pathlib import Path

# stage_01
Path("src/mlProject/pipeline/stage_01_data_ingestion.py").write_text(
    """from mlProject.config.configuration import ConfigurationManager
from mlProject.components.data_ingestion import DataIngestion
from mlProject import logger

STAGE_NAME = 'Data Ingestion stage'

class DataIngestionTrainingPipeline:
    def __init__(self): pass
    def main(self):
        config = ConfigurationManager()
        cfg = config.get_data_ingestion_config()
        di = DataIngestion(config=cfg)
        di.download_file()
        di.extract_zip_file()
"""
)

# stage_02
Path("src/mlProject/pipeline/stage_02_data_validation.py").write_text(
    """from mlProject.config.configuration import ConfigurationManager
from mlProject.components.data_validation import DataValiadtion
from mlProject import logger

STAGE_NAME = 'Data Validation stage'

class DataValidationTrainingPipeline:
    def __init__(self): pass
    def main():
        config = ConfigurationManager()
        cfg = config.get_data_validation_config()
        dv = DataValiadtion(config=cfg)
        dv.validate_all_columns()
"""
)

# stage_03
Path("src/mlProject/pipeline/stage_03_data_transformation.py").write_text(
    """from mlProject.config.configuration import ConfigurationManager
from mlProject.components.data_transformation import DataTransformation
from mlProject import logger
from pathlib import Path

STAGE_NAME = 'Data Transformation stage'

class DataTransformationTrainingPipeline:
    def __init__(self): pass
    def main(self):
        try:
            with open(Path('artifacts/data_validation/status.txt'), 'r') as f:
                status = f.read().split(' ')[-1]
            if status == 'True':
                config = ConfigurationManager()
                cfg = config.get_data_transformation_config()
                dt = DataTransformation(config=cfg)
                dt.train_test_spliting()
            else:
                raise Exception('Data schema is not valid')
        except Exception as e:
            print(e)
"""
)

# stage_04
Path("src/mlProject/pipeline/stage_04_model_trainer.py").write_text(
    """from mlProject.config.configuration import ConfigurationManager
from mlProject.components.model_trainer import ModelTrainer
from mlProject import logger

STAGE_NAME = 'Model Trainer stage'

class ModelTrainerTrainingPipeline:
    def __init__(self): pass
    def main(self):
        config = ConfigurationManager()
        cfg = config.get_model_trainer_config()
        mt = ModelTrainer(config=cfg)
        mt.train()
"""
)

# stage_05
Path("src/mlProject/pipeline/stage_05_model_evaluation.py").write_text(
    """from mlProject.config.configuration import ConfigurationManager
from mlProject.components.model_evaluation import ModelEvaluation
from mlProject import logger

STAGE_NAME = 'Model evaluation stage'

class ModelEvaluationTrainingPipeline:
    def __init__(self): pass
    def main(self):
        config = ConfigurationManager()
        cfg = config.get_model_evaluation_config()
        me = ModelEvaluation(config=cfg)
        me.log_into_mlflow()
"""
)

# prediction
Path("src/mlProject/pipeline/prediction.py").write_text(
    """import joblib
import numpy as np
from pathlib import Path

class PredictionPipeline:
    def __init__(self):
        self.model = joblib.load(Path('artifacts/model_trainer/model.joblib'))
    def predict(self, data):
        return self.model.predict(data)
"""
)
print("✅ All pipeline stage files written.")

✅ All pipeline stage files written.


### 4.10 — `main.py` & `app.py`

In [23]:
main_py = '''
from mlProject import logger
from mlProject.pipeline.stage_01_data_ingestion import DataIngestionTrainingPipeline
from mlProject.pipeline.stage_02_data_validation import DataValidationTrainingPipeline
from mlProject.pipeline.stage_03_data_transformation import DataTransformationTrainingPipeline
from mlProject.pipeline.stage_04_model_trainer import ModelTrainerTrainingPipeline
from mlProject.pipeline.stage_05_model_evaluation import ModelEvaluationTrainingPipeline


STAGE_NAME = "Data Ingestion stage"
try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    DataIngestionTrainingPipeline().main()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\nx==========x")
except Exception as e:
    logger.exception(e); raise e

STAGE_NAME = "Data Validation stage"
try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    DataValidationTrainingPipeline().main()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\nx==========x")
except Exception as e:
    logger.exception(e); raise e

STAGE_NAME = "Data Transformation stage"
try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    DataTransformationTrainingPipeline().main()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\nx==========x")
except Exception as e:
    logger.exception(e); raise e

STAGE_NAME = "Model Trainer stage"
try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    ModelTrainerTrainingPipeline().main()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\nx==========x")
except Exception as e:
    logger.exception(e); raise e

STAGE_NAME = "Model evaluation stage"
try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    ModelEvaluationTrainingPipeline().main()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\nx==========x")
except Exception as e:
    logger.exception(e); raise e
'''

app_py = '''
from flask import Flask, render_template, request
import os
import numpy as np
from mlProject.pipeline.prediction import PredictionPipeline

app = Flask(__name__)

@app.route(\'/\', methods=[\'GET\'])
def homePage():
    return render_template("index.html")

@app.route(\'/train\', methods=[\'GET\'])
def training():
    os.system("python main.py")
    return "Training Successful!"

@app.route(\'/predict\', methods=[\'POST\', \'GET\'])
def index():
    if request.method == \'POST\':
        try:
            fixed_acidity        = float(request.form[\'fixed_acidity\'])
            volatile_acidity     = float(request.form[\'volatile_acidity\'])
            citric_acid          = float(request.form[\'citric_acid\'])
            residual_sugar       = float(request.form[\'residual_sugar\'])
            chlorides            = float(request.form[\'chlorides\'])
            free_sulfur_dioxide  = float(request.form[\'free_sulfur_dioxide\'])
            total_sulfur_dioxide = float(request.form[\'total_sulfur_dioxide\'])
            density              = float(request.form[\'density\'])
            pH                   = float(request.form[\'pH\'])
            sulphates            = float(request.form[\'sulphates\'])
            alcohol              = float(request.form[\'alcohol\'])

            data = [fixed_acidity, volatile_acidity, citric_acid, residual_sugar,
                    chlorides, free_sulfur_dioxide, total_sulfur_dioxide,
                    density, pH, sulphates, alcohol]
            data = np.array(data).reshape(1, 11)

            obj = PredictionPipeline()
            predict = obj.predict(data)
            return render_template(\'results.html\', prediction=str(predict))
        except Exception as e:
            print(\'Exception:\', e)
            return \'something is wrong\'
    else:
        return render_template(\'index.html\')

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8080, debug=True)
'''

with open("main.py", "w") as f:
    f.write(main_py.strip())

with open("app.py", "w") as f:
    f.write(app_py.strip())

print("✅ main.py and app.py written.")

✅ main.py and app.py written.


### 4.11 — Install `mlProject` as a package

In [34]:
!pip install -e . -q
print("✅ mlProject package installed.")

  Preparing metadata (setup.py) ... done
✅ mlProject package installed.


In [35]:
!pip install -e . -q
import importlib, sys

# Force reload in case kernel cached old path
if 'mlProject' in sys.modules:
    del sys.modules['mlProject']

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "src"))
print("✅ Ready.")

  Preparing metadata (setup.py) ... done
✅ Ready.


---
## 🚀 Section 5 — Run Training Pipeline

Runs all 5 stages sequentially.

### Stage 1 — Data Ingestion
Downloads the Drinks dataset zip from GitHub and extracts it.

In [37]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

from mlProject import logger
from mlProject.pipeline.stage_01_data_ingestion import DataIngestionTrainingPipeline

STAGE_NAME = "Data Ingestion stage"
try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    DataIngestionTrainingPipeline().main()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<")
except Exception as e:
    logger.exception(e)
    raise e


### Stage 2 — Data Validation
Validates that all columns in the CSV match `schema.yaml`.

In [38]:
from mlProject import logger
from mlProject.pipeline.stage_02_data_validation import DataValidationTrainingPipeline

# Imports needed for the temporary class redefinition
from mlProject.config.configuration import ConfigurationManager
from mlProject.components.data_validation import DataValiadtion

# Temporarily redefine the class with the corrected method signature
class DataValidationTrainingPipeline:
    def __init__(self): pass
    def main(self): # Added 'self' here
        config = ConfigurationManager()
        cfg = config.get_data_validation_config()
        dv = DataValiadtion(config=cfg)
        dv.validate_all_columns()

STAGE_NAME = "Data Validation stage"
try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    DataValidationTrainingPipeline().main()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<")
except Exception as e:
    logger.exception(e)
    raise e

with open("artifacts/data_validation/status.txt") as f:
    print("Validation result:", f.read())

Validation result: Validation status: True


### Stage 3 — Data Transformation
Performs train/test split (75/25) and saves `train.csv` and `test.csv`.

In [39]:
from mlProject import logger
from mlProject.pipeline.stage_03_data_transformation import DataTransformationTrainingPipeline

STAGE_NAME = "Data Transformation stage"
try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    DataTransformationTrainingPipeline().main()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<")
except Exception as e:
    logger.exception(e)
    raise e


(1199, 12)
(400, 12)


### Stage 4 — Model Trainer
Trains an `ElasticNet` model with hyperparameters from `params.yaml` and saves `model.joblib`.

In [40]:
from mlProject import logger
from mlProject.pipeline.stage_04_model_trainer import ModelTrainerTrainingPipeline

STAGE_NAME = "Model Trainer stage"
try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    ModelTrainerTrainingPipeline().main()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<")
except Exception as e:
    logger.exception(e)
    raise e

import os
print("Model saved:", os.path.exists("artifacts/model_trainer/model.joblib"))


Model saved: True


### Stage 5 — Model Evaluation
Evaluates the trained model and logs metrics (RMSE, MAE, R²) to MLflow/DagsHub.

In [46]:
from mlProject import logger
import importlib
import sys
import os

# --- Aggressively clear mlProject related modules from sys.modules
# This ensures a clean state for importing and reloading
modules_to_delete = [m for m in sys.modules if m.startswith('mlProject')]
for m in modules_to_delete:
    del sys.modules[m]

# --- Ensure 'src' is in sys.path just in case
current_dir = os.getcwd()
src_path = os.path.join(current_dir, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# --- Now, import the pipeline class and necessary components freshly
# Since we cleared sys.modules, these will be fresh imports from disk
from mlProject.pipeline.stage_05_model_evaluation import ModelEvaluationTrainingPipeline
from mlProject.components.model_evaluation import ModelEvaluation
from mlProject.config.configuration import ConfigurationManager

STAGE_NAME = "Model evaluation stage"

try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    # The pipeline class will now be the freshly imported one
    ModelEvaluationTrainingPipeline().main()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<")
except Exception as e:
    logger.exception(e)
    raise e

import json
with open("artifacts/model_evaluation/metrics.json") as f:
    metrics = json.load(f)
print("""
📊 Model Evaluation Metrics""")
print(f"   RMSE : {metrics['rmse']:.4f}")
print(f"   MAE  : {metrics['mae']:.4f}")
print(f"   R²   : {metrics['r2']:.4f}")

2026/03/31 04:33:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/31 04:33:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'ElasticnetModel'.
2026/03/31 04:33:58 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModel, version 1
Created version '1' of model 'ElasticnetModel'.


🏃 View run bedecked-lynx-671 at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/0/runs/286c8e94312d4bf688d4e932fb2da976
🧪 View experiment at: https://dagshub.com/prithusarkar90/networksecurity.mlflow/#/experiments/0

📊 Model Evaluation Metrics
   RMSE : 0.7079
   MAE  : 0.5675
   R²   : 0.2266


---
## 🔮 Section 6 — Prediction (Inference)

Run a sample prediction using the trained model.

In [47]:
import numpy as np
from mlProject.pipeline.prediction import PredictionPipeline

# Sample input: [fixed_acidity, volatile_acidity, citric_acid, residual_sugar,
#                chlorides, free_sulfur_dioxide, total_sulfur_dioxide,
#                density, pH, sulphates, alcohol]
sample = np.array([[7.4, 0.70, 0.00, 1.9, 0.076,
                    11.0, 34.0, 0.9978, 3.51, 0.56, 9.4]])

pipeline = PredictionPipeline()
prediction = pipeline.predict(sample)

print(f"🍷 Predicted Quality Score: {prediction[0]:.4f}")

🍷 Predicted Quality Score: 5.2605


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but ElasticNet was fitted with feature names
  warnings.warn(


---
## 🌐 Section 7 — Flask App (Optional)

> ⚠️ The Flask server runs in the background. Use `ngrok` or Colab's port-forwarding to access it.
> Run only if you need the web UI — it will block the cell unless backgrounded.

In [ ]:
# ── Write HTML Templates ──────────────────────────────────────────────────────
index_html = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Drinks Quality Checker</title>
  <style>
    body { font-family: Arial, sans-serif; max-width: 600px; margin: 40px auto; padding: 0 20px; }
    h1   { color: #4a3728; }
    input[type=number] { width: 100%; padding: 8px; margin: 4px 0 12px; box-sizing: border-box; }
    input[type=submit] { background: #6b3a2a; color: white; padding: 10px 24px;
                         border: none; cursor: pointer; border-radius: 4px; font-size: 16px; }
  </style>
</head>
<body>
  <h1>🍷 Drinks Quality Checker</h1>
  <form action="/predict" method="POST">
    <label>Fixed Acidity</label><input type="number" step="any" name="fixed_acidity" required>
    <label>Volatile Acidity</label><input type="number" step="any" name="volatile_acidity" required>
    <label>Citric Acid</label><input type="number" step="any" name="citric_acid" required>
    <label>Residual Sugar</label><input type="number" step="any" name="residual_sugar" required>
    <label>Chlorides</label><input type="number" step="any" name="chlorides" required>
    <label>Free Sulfur Dioxide</label><input type="number" step="any" name="free_sulfur_dioxide" required>
    <label>Total Sulfur Dioxide</label><input type="number" step="any" name="total_sulfur_dioxide" required>
    <label>Density</label><input type="number" step="any" name="density" required>
    <label>pH</label><input type="number" step="any" name="pH" required>
    <label>Sulphates</label><input type="number" step="any" name="sulphates" required>
    <label>Alcohol</label><input type="number" step="any" name="alcohol" required>
    <br><input type="submit" value="Predict Quality">
  </form>
</body>
</html>"""

results_html = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Prediction Result</title>
  <style>
    body   { font-family: Arial, sans-serif; max-width: 500px; margin: 80px auto; text-align: center; }
    .score { font-size: 64px; color: #6b3a2a; font-weight: bold; }
    a      { color: #6b3a2a; }
  </style>
</head>
<body>
  <h1>🍷 Predicted Quality</h1>
  <p class="score">{{ prediction }}</p>
  <p><a href="/predict">← Try another</a></p>
</body>
</html>"""

with open("templates/index.html",   "w") as f: f.write(index_html)
with open("templates/results.html", "w") as f: f.write(results_html)

print("✅ HTML templates written.")

In [ ]:
# ── Optional: Install pyngrok and expose the app ──────────────────────────────
# Uncomment the lines below to run the Flask app with a public URL via ngrok

# !pip install -q pyngrok
# from pyngrok import ngrok
# import threading
# import subprocess

# ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))  # Add NGROK_TOKEN to Colab Secrets

# def run_flask():
#     subprocess.run(["python", "app.py"])

# thread = threading.Thread(target=run_flask, daemon=True)
# thread.start()

# public_url = ngrok.connect(8080)
# print(f"\n🌐 Flask App running at: {public_url}")

print("ℹ️  Flask cell is ready — uncomment above to launch with ngrok.")

---
## 📋 Section 8 — Artifacts Summary

In [48]:
import os
import json

print("=" * 50)
print("📦 ARTIFACTS SUMMARY")
print("=" * 50)

for root, dirs, files in os.walk("artifacts"):
    level = root.replace("artifacts", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for file in files:
        filepath = os.path.join(root, file)
        size_kb = os.path.getsize(filepath) / 1024
        print(f"{sub_indent}📄 {file}  ({size_kb:.1f} KB)")

print()
print("=" * 50)
print("📊 FINAL METRICS")
print("=" * 50)
with open("artifacts/model_evaluation/metrics.json") as f:
    m = json.load(f)
print(f"  RMSE : {m['rmse']:.4f}")
print(f"  MAE  : {m['mae']:.4f}")
print(f"  R²   : {m['r2']:.4f}")

📦 ARTIFACTS SUMMARY
📁 artifacts/
  📁 model_evaluation/
    📄 metrics.json  (0.1 KB)
  📁 data_ingestion/
    📄 Drinks-data.csv  (98.6 KB)
    📄 data.zip  (22.8 KB)
  📁 data_transformation/
    📄 test.csv  (22.9 KB)
    📄 train.csv  (68.6 KB)
  📁 data_validation/
    📄 status.txt  (0.0 KB)
  📁 model_trainer/
    📄 model.joblib  (1.2 KB)

📊 FINAL METRICS
  RMSE : 0.7079
  MAE  : 0.5675
  R²   : 0.2266


In [49]:
# ── Download all artifacts as a zip ──────────────────────────────────────────
import shutil
from google.colab import files

shutil.make_archive("artifacts", "zip", ".", "artifacts")
files.download("artifacts.zip")
print("✅ artifacts.zip downloaded.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ artifacts.zip downloaded.
